In [51]:
import torch
import torchaudio

In [52]:
AUDIO_PATH = './sample.wav'
audio, sample_rate = torchaudio.load(AUDIO_PATH)
print(audio, sample_rate)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]) 48000


In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [54]:
print(audio.ndim) # 2 = dual channel, 1 = single channel

2


In [55]:
# if audio.ndim > 1:
#     audio = audio.mean(audio, keepdim=True)
    

In [56]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
print("labels: ", bundle.get_labels())
print("sample rate: ", bundle.sample_rate)

labels:  ('-', '|', 'E', 'T', 'A', 'O', 'N', 'I', 'H', 'S', 'R', 'D', 'L', 'U', 'M', 'W', 'C', 'F', 'G', 'Y', 'P', 'B', 'V', 'K', "'", 'X', 'J', 'Q', 'Z')
sample rate:  16000


In [57]:
if sample_rate != bundle.sample_rate:
    audio = torchaudio.functional.resample(audio, sample_rate, bundle.sample_rate)

In [58]:
print(device)

cpu


In [59]:
model = bundle.get_model().to(device)

with torch.inference_mode():
    emission, _ = model(audio)
    
print(emission)
print(emission.shape)

tensor([[[13.4189,  0.7390, -3.8436,  ..., -5.9796, -6.9032, -7.8908],
         [13.3226,  0.7736, -3.8760,  ..., -5.8060, -6.8356, -7.7062],
         [13.9044,  0.9230, -3.8635,  ..., -6.1372, -7.0591, -7.9493],
         ...,
         [13.1806,  1.3941, -3.5958,  ..., -6.1341, -6.9793, -7.7255],
         [12.6044,  0.1752, -4.4410,  ..., -4.6929, -6.1534, -7.2614],
         [13.3286,  1.5009, -3.6989,  ..., -6.0438, -7.1270, -7.7222]],

        [[13.2140,  0.9174, -4.0836,  ..., -5.7193, -6.8809, -7.8263],
         [13.3787,  0.9517, -4.0560,  ..., -5.7715, -6.9809, -7.8303],
         [13.6677,  1.0070, -4.0849,  ..., -5.8930, -7.0361, -7.9109],
         ...,
         [12.9776,  1.4603, -3.7904,  ..., -5.7771, -6.9236, -7.5996],
         [12.4188,  0.3815, -4.5377,  ..., -4.5201, -6.1221, -7.2309],
         [12.4189,  0.3816, -4.5379,  ..., -4.5203, -6.1224, -7.2306]]])
torch.Size([2, 53, 29])


In [60]:
class GreedyCTC(torch.nn.Module):
    def __init__(self, labels, blank=0): # blank isi index dari '-' di tuple labels
        super().__init__()
        self.labels = labels
        self.blank = blank
    
    def forward(self, emission):
        indexs = torch.argmax(emission, dim = -1)
        indexs = torch.unique_consecutive(indexs, dim = -1)
        indexs = [i for i in indexs if i != self.blank]
        return "".join([self.labels[i] for i in indexs])

In [61]:
ctc = GreedyCTC(bundle.get_labels())
prediction = ctc(emission[0]) # msh dual channel, jd pass salah satu channel dgn cara [0] atau [1]. kl ga salah [0] = left
print(prediction)
print(prediction.replace("|", " "))
prediction = ctc(emission[1]) # [1] = right
print(prediction)
print(prediction.replace("|", " "))

WHAT|OUT|
WHAT OUT 
WHAT|OUT|
WHAT OUT 


In [ ]:
def predict_text():
    